In [5]:
from datetime import datetime
from enum import Enum
import re
import csv
import json

class OperationStatus(Enum):
    success = 0
    fail = 1

class Operation:
    
    def __init__(self, type : str, amount : int, datetime: datetime, balance : int, status : OperationStatus) -> None:
        self.type = type
        self.amount = amount
        self.datetime = datetime
        self.balance = balance
        self.status = status

    def __str__(self) -> str:
        return f"{self.datetime}: operation type: {self.type}, amount: {self.amount}, balance: {self.balance}"

class Account:
    _account_counter = 1000

    def __init__(self, account_holder: str, balance : int = 0) -> None:
        if balance < 0:
            raise ValueError('balance must be positive')

        try:
            self.validate_holder(account_holder)
        except ValueError as e:
            raise e

        self.holder = account_holder
        self._balance = balance

        self.operations_history = []

        self.account_number = f'ACC-{Account._account_counter}'
        Account._account_counter += 1
        pass

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")

        operation_type = 'deposit'
        
        self._balance += amount
        self.operations_history.append(
            Operation(
                type=operation_type, 
                amount=amount, 
                balance=self._balance, 
                status=OperationStatus.success,
                datetime=datetime.now()
            ))

        

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError('amount must be positive')

        operation_type = 'withdraw'
        if self._balance < amount:
            self.operations_history.append(
                Operation(
                    type=operation_type,
                    amount=amount,
                    balance=self._balance,
                    status=OperationStatus.fail,
                    datetime=datetime.now()
                )
            )
            return

        self._balance -= amount
        self.operations_history.append(
            Operation(
                type=operation_type,
                amount=amount,
                balance=self._balance,
                status=OperationStatus.success,
                datetime=datetime.now()
            )
        )
    def get_balance(self):
        return self._balance
    
    def get_history(self) -> list[Operation]:
        return self.operations_history.copy()

# task 2

    @staticmethod
    def validate_holder(name):
        if not re.fullmatch(r"^[A-ZА-ЯЁ][a-zа-яё]+ [A-ZА-ЯЁ][a-zа-яё]+$", name):
            raise ValueError('name must be in format like "Imanov Rakhman", "Иванов Иван" and etc')

    def get_large_operations(self, n = 5):
        return sorted(self.operations_history, key=lambda x: x.amount, reverse=True)[:n]

# task 3

    def load_history(self, filename):

        if filename.endswith('.json'):
            transactions = self._load_json(filename)

        elif filename.endswith('.csv'):
            transactions = self._load_csv(filename)

        else:
            raise ValueError('unsupported file format')

        transactions = self.clean_history(transactions)

        self.operations_history.clear()
        self._balance = 0

        for transaction in transactions:

            operation = transaction['operation']
            amount = transaction['amount']

            if operation in ('deposit', 'interest'):
                self._balance += amount

            elif operation == 'withdraw':
                self._balance -= amount

            self.operations_history.append(
                Operation(
                    type=operation,
                    amount=amount,
                    datetime=datetime.now(),
                    balance=self._balance,
                    status=OperationStatus.success
                )
            )


    def _load_json(self, filename):

        with open(filename, 'r', encoding='utf-8') as file:
            transactions = json.load(file)

        return [
            transaction
            for transaction in transactions
            if transaction['account_number'] == self.account_number
        ]


    def _load_csv(self, filename):

        with open(filename, newline='', encoding='utf-8') as file:
            reader = csv.DictReader(file)

            transactions = list(reader)

        for transaction in transactions:

            if transaction['amount']:
                transaction['amount'] = float(transaction['amount'])
            else:
                transaction['amount'] = None

        return [
            transaction
            for transaction in transactions
            if transaction['account_number'] == self.account_number
        ]


    def clean_history(self, transactions):

        valid_transactions = []

        for t in transactions:

            operation = t.get('operation')
            amount = t.get('amount')
            date = t.get('date')
            status = t.get('status')

            if operation not in self.allowed_operations:
                continue

            if amount is None or amount <= 0:
                continue

            if status != 'success':
                continue

            valid_date = False

            for fmt in (
                "%Y-%m-%d %H:%M:%S",
                "%d/%m/%Y %H:%M"
            ):
                try:
                    datetime.strptime(date, fmt)
                    valid_date = True
                    break

                except ValueError:
                    pass

            if not valid_date:
                continue

            valid_transactions.append(t)

        return valid_transactions

In [6]:
acc = Account("Ivan Ivanov", 1000)

acc.withdraw(200)
acc.deposit(500)
# acc.withdraw(5000)

[print(i) for i in acc.get_history()]
print(acc.get_balance(), acc._account_counter)
[print(i) for i in acc.get_large_operations()]

2026-10-04 21:48:05.480303: operation type: withdraw, amount: 200, balance: 800
2026-10-04 21:48:05.480330: operation type: deposit, amount: 500, balance: 1300
1300 1001
2026-10-04 21:48:05.480330: operation type: deposit, amount: 500, balance: 1300
2026-10-04 21:48:05.480303: operation type: withdraw, amount: 200, balance: 800


[None, None]

In [7]:
class CheckingAccount(Account):
    account_type = 'checking'
    allowed_operations = {
        "deposit",
        "withdraw"
    }

class SavingsAccount(Account):
    account_type = 'savings'
    allowed_operations = {
        "deposit",
        "withdraw",
        "interest"
    }
    
    def apply_interest(self, rate = 0.07):
        if not (0 < rate < 1):
            raise ValueError('rate must be in between 0 and 1')

        self.deposit(self._balance * rate)

    def withdraw(self, amount):
        if amount > (self._balance * 0.5):
            raise ValueError("amount must be not greater than half of balance")

        return super().withdraw(amount)
    

In [9]:
acc = CheckingAccount("Ivan Ivanov")
acc.account_number = "ACC-100001"
acc.load_history("transactions_dirty.json")
[print(i) for i in acc.get_history()]

2026-10-04 21:49:05.865401: operation type: deposit, amount: 921.0, balance: 921.0
2026-10-04 21:49:05.865490: operation type: deposit, amount: 607.0, balance: 1528.0
2026-10-04 21:49:05.865492: operation type: deposit, amount: 488.0, balance: 2016.0
2026-10-04 21:49:05.865494: operation type: deposit, amount: 129.0, balance: 2145.0
2026-10-04 21:49:05.865495: operation type: deposit, amount: 880.0, balance: 3025.0
2026-10-04 21:49:05.865497: operation type: withdraw, amount: 352.0, balance: 2673.0
2026-10-04 21:49:05.865499: operation type: withdraw, amount: 65.0, balance: 2608.0
2026-10-04 21:49:05.865500: operation type: withdraw, amount: 245.0, balance: 2363.0
2026-10-04 21:49:05.865501: operation type: deposit, amount: 327.0, balance: 2690.0
2026-10-04 21:49:05.865502: operation type: withdraw, amount: 333.0, balance: 2357.0
2026-10-04 21:49:05.865503: operation type: withdraw, amount: 198.0, balance: 2159.0
2026-10-04 21:49:05.865504: operation type: deposit, amount: 838.0, balan

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]